# Environment Setup
## Library Imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import timm
import wandb
from kaggle_secrets import UserSecretsClient
from torchvision.models import resnet18
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings("ignore")

## Weights and Biases Configuration Setup

In [ ]:
user_secrets = UserSecretsClient()

os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

os.environ["WANDB_DISABLE_SERVICE"] = "true"
os.environ["WANDB_START_METHOD"] = "thread"

try:
    wandb.login()
    WANDB_MODE = "online"
except:
    WANDB_MODE = "disabled" 
    print("Running in offline mode - WandB logging disabled")

wandb.login()
WANDB_PROJECT = "23f3003805-t12026"

## Device Configuration

In [ ]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data Configuration
## Loading data

In [ ]:
DATA_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"


STEMS_DIR = f"{DATA_DIR}/genres_stems"
MASHUPS_DIR = f"{DATA_DIR}/mashups"
ESC50_DIR = f"{DATA_DIR}/ESC-50-master/audio"

TEST_CSV = f"{DATA_DIR}/test.csv"
SUBMISSION_CSV = f"{DATA_DIR}/sample_submission.csv"

## Genre Label Mapping

In [ ]:
GENRES = sorted(os.listdir(STEMS_DIR))
GENRE_TO_IDX = {g:i for i,g in enumerate(GENRES)}
# jazz=0 rock=1 ...

IDX_TO_GENRE = {i:g for g,i in GENRE_TO_IDX.items()}
EPOCHS=40

print(GENRES)

# Dataset Construction
## Stem Metadata Creation

In [ ]:
records = []
# stores path to 4 stems for all songs

for genre in GENRES:
    genre_path = os.path.join(STEMS_DIR, genre)

    for song in os.listdir(genre_path):

        song_path = os.path.join(genre_path, song)

        stems = {
            "bass": os.path.join(song_path,"bass.wav"),
            "drums": os.path.join(song_path,"drums.wav"),
            "other": os.path.join(song_path,"other.wav"),
            "vocals": os.path.join(song_path,"vocals.wav")
        }

        records.append({
            "genre":genre,
            "genre_id":GENRE_TO_IDX[genre],
            "stems":stems
        })

df = pd.DataFrame(records)
df.head()

# Exploratory Data Analysis Part 1

In [ ]:
print("Total samples:", len(df))
print("Total genres:", len(GENRES))
print("Genres:", GENRES)

df.head()

## Genre Distribution

In [ ]:
genre_counts = df['genre'].value_counts()

#helps detect class imbalance

plt.figure(figsize=(10,5))
genre_counts.plot(kind='bar')
plt.title("Genre Distribution")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

print(genre_counts)

## File Size Analysis

In [ ]:

LESS_THAN_4KB = 0
LESS_THAN_5MB = 0

SIZE_4KB = 4 * 1024
SIZE_5MB = 5 * 1024 * 1024

for genre in GENRES:
    genre_path = os.path.join(STEMS_DIR, genre)

    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)

        for stem in ["bass.wav","drums.wav","other.wav","vocals.wav"]:
            stem_path = os.path.join(song_path, stem)

            if not os.path.exists(stem_path):
                continue

            size = os.path.getsize(stem_path)

            if size < SIZE_4KB:
                LESS_THAN_4KB += 1

            if size < SIZE_5MB:
                LESS_THAN_5MB += 1

print("Files < 4KB:", LESS_THAN_4KB)
print("Files < 5MB:", LESS_THAN_5MB)

## ESC-50 Noise Dataset

In [ ]:
noise_files = []
# stores noise

for f in os.listdir(ESC50_DIR):
    if f.endswith(".wav"):
        noise_files.append(os.path.join(ESC50_DIR,f))

print("Noise files:",len(noise_files))

# Data Preprocessing and Feature Engineering
## Audio Processing Functions
### Audio Loading Function

In [ ]:
def load_audio(path, sr=22050):

    y, _ = librosa.load(path, sr=sr)

    return y

### Synthetic Mashup Generator

In [ ]:
genre_groups = {g: df[df.genre == g] for g in GENRES}

def synthetic_mashup(genre):
    # takes 4 random songs of same genre nad mixes stems
    
    songs = genre_groups[genre].sample(4)

    bass = load_audio(songs.iloc[0].stems["bass"])
    drums = load_audio(songs.iloc[1].stems["drums"])
    vocals = load_audio(songs.iloc[2].stems["vocals"])
    other = load_audio(songs.iloc[3].stems["other"])

    min_len = min(len(bass), len(drums), len(vocals), len(other))
    # align to shortest length

    bass = bass[:min_len]
    drums = drums[:min_len]
    vocals = vocals[:min_len]
    other = other[:min_len]

    w = np.random.uniform(0.5, 1.5, 4)
    # ransdom weight for mixing

    mix = (
        w[0]*bass +
        w[1]*drums +
        w[2]*vocals +
        w[3]*other
    )

    mix = mix / (np.max(np.abs(mix)) + 1e-6)
    # normalize

    return mix

## Silence Analysis

In [ ]:
# defines how is silence spread out across dtaa
def compute_silence(y, threshold=0.01):
    silent = np.where(np.abs(y) < threshold, 1, 0)
    silence_ratio = np.mean(silent)
    return silence_ratio

silence_data = []

for i,row in tqdm(df.iterrows(), total=len(df)):
    y = synthetic_mashup(row.genre)
    
    silence_ratio = compute_silence(y)
    
    silence_data.append({
        "Genre": row.genre,
        "Silence_Ratio": silence_ratio
    })

df_silence = pd.DataFrame(silence_data)

df_silence.head()

## Silence Distribution

In [ ]:

plt.figure(figsize=(8,4))
plt.hist(df_silence["Silence_Ratio"], bins=30)
plt.title("Silence Ratio Distribution")
plt.xlabel("Silence Ratio")
plt.ylabel("Frequency")
plt.show()

print("Average Silence:", df_silence["Silence_Ratio"].mean())

## Audio Data Augmentation Techniques
### Noise Injection

In [ ]:
import torchaudio.transforms as T

spec_augment = torch.nn.Sequential(
    T.FrequencyMasking(freq_mask_param=15),
    # blocks certian freq channels
    T.TimeMasking(time_mask_param=35)
    # blocks time steps
)

In [ ]:
def add_noise(signal, snr_db=10):
    # adds random noise 
    noise_path = random.choice(noise_files)
    noise = load_audio(noise_path)

    if len(noise) < len(signal):
        noise = np.tile(noise, len(signal)//len(noise)+1)

    noise = noise[:len(signal)]

    signal_power = np.mean(signal**2)
    noise_power = np.mean(noise**2)

    factor = np.sqrt(signal_power/(10**(snr_db/10)*noise_power))
    # checks power ratio for addingnoice at correct decibel
    noisy = signal + factor*noise

    return noisy

### Time Stretching and Padding

In [ ]:
def augment_audio(y, target_len=None):
# stretching for avoiding overfiiting
    if random.random() < 0.5:
        rate = random.uniform(0.9, 1.1)
        y = librosa.effects.time_stretch(y, rate=rate)

    if random.random() < 0.5:
        y = add_noise(y, snr_db=random.randint(5, 20))

    if target_len is not None:
    # standardize length
        if len(y) > target_len:
            y = y[:target_len]

        else:
            pad = target_len - len(y)
            y = np.pad(y, (0, pad))

    return y

### Random Audio Cropping

In [ ]:
def random_crop(y, crop_size):
# randomly crops audio to specific length
    if len(y) <= crop_size:
        return np.pad(y, (0, crop_size - len(y)))

    start = random.randint(0, len(y) - crop_size)

    return y[start:start + crop_size]

# Exploratory Data Analysis Part 2
## Sample Waveform Visualization

In [ ]:
# amplitude vs graph
# peaks = loud sound flat=silence 
sample_row = df.sample(1).iloc[0]

audio = synthetic_mashup(sample_row.genre)

plt.figure(figsize=(12,4))
plt.plot(audio)
plt.title(f"Waveform Example - Genre: {sample_row.genre}")
plt.show()

## Spectrogram Visualization

In [ ]:
# audio into freq vs time
# cnn uses spectro
# bright=strong freq dark=weak
S = librosa.feature.melspectrogram(y=audio, sr=22050)
S_db = librosa.power_to_db(S, ref=np.max)

plt.figure(figsize=(10,4))
librosa.display.specshow(S_db, sr=22050, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title("Mel Spectrogram")
plt.show()

## Audio Length Analysis

In [ ]:
lengths = []
# audio files can have diff length

for i,row in tqdm(df.iterrows(), total=len(df)):
    y = synthetic_mashup(row.genre)
    lengths.append(len(y)/22050)  # seconds

plt.figure(figsize=(8,4))
plt.hist(lengths, bins=30)
plt.title("Audio Length Distribution (seconds)")
plt.xlabel("Seconds")
plt.ylabel("Frequency")
plt.show()

print("Min length:", np.min(lengths))
print("Max length:", np.max(lengths))
print("Mean length:", np.mean(lengths))

# Dataset & DataLoader
## Custom PyTorch Dataset

In [ ]:
class StemDataset(Dataset):

    def __init__(self, df, sr=22050):
        self.df = df
        self.sr = sr

    def __len__(self):
        return len(self.df)

    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = synthetic_mashup(row.genre)
        
        y = random_crop(y, 22050 * 6)
        # cuts random 6 sec segment
        y = augment_audio(y, 22050 * 6)
        # add noise for shifting pitch 
        
        # y = y / (np.max(np.abs(y)) + 1e-6)
        y = torch.tensor(y).float().unsqueeze(0)
        label = row.genre_id
        return y, label

## Custom Collate Function

In [ ]:
def collate_fn(batch):
    waves = [b[0] for b in batch]
    labels = [b[1] for b in batch]
    # pads the waves and 
    max_len = max([w.shape[1] for w in waves])
    padded = []
    for w in waves:
        pad = max_len - w.shape[1]
        padded.append(nn.functional.pad(w,(0,pad)))

    return torch.stack(padded), torch.tensor(labels)

## Train-Validation Split

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df.genre
)

## DataLoaders

In [ ]:
train_ds = StemDataset(train_df)
val_ds = StemDataset(val_df)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# Spectrogram Feature Transformation
## Mel Spectrogram Conversion

In [ ]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=22050,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
    f_min=20,
    f_max=11025
)
# converts raw audio into mel frequency

# Feature Engineering — MFCC Baseline
## MFCC Feature Extraction

In [ ]:
def extract_mfcc_from_stems(row):
    y = synthetic_mashup(row.genre)
    y = random_crop(y, 22050 * 6)

    mfcc = librosa.feature.mfcc(y=y, sr=22050, n_mfcc=40)

    return mfcc.mean(axis=1)

print("Extracting MFCC features...")

## MFCC Dataset Generation

In [ ]:
X = []
y = []

for i,row in tqdm(df.iterrows(), total=len(df)):
    
    feat = extract_mfcc_from_stems(row)
    
    X.append(feat)
    y.append(row.genre_id)

X = np.array(X)
y = np.array(y)

X_train,X_val,y_train,y_val = train_test_split(
    X,y,test_size=0.1,stratify=y
)

# Model Architectures and Training
## Logistic Regression (Baseline)

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="logistic_regression_baseline",
    reinit=True, 
    config={
        "model": "LogisticRegression",
        "features": "MFCC",
        "max_iter": 2000
    }
)

### Training Logistic Regression

In [ ]:
clf = LogisticRegression(max_iter=2000)

clf.fit(X_train,y_train)

pred = clf.predict(X_val)

f1 = f1_score(y_val,pred,average="macro")
acc = accuracy_score(y_val,pred)

run.log({
    "F1_score":f1,
    "accuracy":acc
})

print("Logistic Regression")
print("F1:",f1)
print("Accuracy:",acc)

run.finish()

## CNN Model
### CNN Architecture

In [ ]:
class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()
        self.mel = mel_transform
        self.net = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d(1)
            # squashes entire 2d data to spectrogram into singel vector
        )
        self.fc = nn.Linear(128,len(GENRES))

    
    def forward(self,x):
        x = self.mel(x)
        x = torch.log(x + 1e-6)
        # convert amplitudes to decibels

        # if self.training:
            # x = spec_augment(x)

        x = self.net(x)
        x = x.view(x.size(0),-1)
        return self.fc(x)

### CNN Training

In [ ]:
run=wandb.init(
    project=WANDB_PROJECT,
    name="simple_cnn",
    reinit=True,
    config={
        "model":"SimpleCNN",
        "optimizer":"Adam",
        "lr":1e-3,
        "epochs":10,
        "batch_size":64
    }
)

In [ ]:
cnn_model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(cnn_model.parameters(),lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    cnn_model.train()
    total_loss = 0

    for x,y in tqdm(train_loader):
        x,y = x.to(device),y.to(device)
        optimizer.zero_grad()
        preds = cnn_model(x)
        
        loss = criterion(preds,y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        
    avg_loss = total_loss/len(train_loader)
    run.log({
        "epoch":epoch,
        "train_loss":avg_loss
    })

    print("Epoch",epoch,"Loss",avg_loss)

### CNN Evaluation

In [ ]:
def evaluate_model(model):
    model.eval()
    
    preds_all=[]
    labels_all=[]

    with torch.no_grad():
        
        for x,y in val_loader:
            x=x.to(device)
            out=model(x)

            preds=out.argmax(1).cpu().numpy()

            preds_all.extend(preds)
            labels_all.extend(y.numpy())

    f1=f1_score(labels_all,preds_all,average="macro")
    acc=accuracy_score(labels_all,preds_all)

    return f1,acc

In [ ]:
f1, acc = evaluate_model(cnn_model)

run.log({
    "epoch": epoch,
    "train_loss": avg_loss,
    "val_f1": f1,
    "val_accuracy": acc
})

print("CNN F1:", f1)
print("CNN Accuracy:", acc)

run.finish()

## CRNN Model
### CRNN Architecture

In [ ]:
class CRNN(nn.Module):

    def __init__(self):
        super().__init__()
        self.mel = mel_transform
        self.cnn = nn.Sequential(
# filterskernal slides over -> detects patterns-> non linearrity-> dim reducion
            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        # extracts local features 
        
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        # remembers what happended in beiginning 
        self.fc = nn.Linear(256,len(GENRES))

    
    def forward(self,x):
        x = self.mel(x)
        x = torch.log(x+1e-6)

        if self.training:
            x = spec_augment(x)

        x = self.cnn(x)
        x = x.mean(dim=2)
        x = x.permute(0,2,1)
        x,_ = self.gru(x)
        x = x.mean(dim=1)

        return self.fc(x)

### CRNN Training

In [ ]:
run=wandb.init(
    project=WANDB_PROJECT,
    name="crnn_model",
    reinit=True,
    config={
        "model":"CRNN",
        "optimizer":"Adam",
        "lr":3e-4,
        "epochs":12
    }
)

In [ ]:
crnn_model = CRNN().to(device)
optimizer = torch.optim.Adam(crnn_model.parameters(),lr=3e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(12):
    crnn_model.train()
    total_loss = 0

    for x,y in tqdm(train_loader):
        x,y = x.to(device),y.to(device)
        optimizer.zero_grad()

        preds = crnn_model(x)
        loss = criterion(preds,y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    run.log({
        "epoch":epoch,
        "train_loss":total_loss/len(train_loader),
        "learning_rate":optimizer.param_groups[0]['lr']
    })

    print("Epoch",epoch,"Loss",total_loss/len(train_loader))

### CRNN Evaluation

In [ ]:
f1, acc = evaluate_model(crnn_model)

run.log({
    "F1_score": f1,
    "accuracy": acc
})

print("CRNN F1:", f1)
print("CRNN Accuracy:", acc)

run.finish()

## EfficientNet Spectrogram Model
### EfficientNet Architecture

In [ ]:
class EfficientNetAudio(nn.Module):

    def __init__(self):
        super().__init__()

        self.mel = mel_transform
        self.freq_mask = torchaudio.transforms.FrequencyMasking(24)
        self.time_mask = torchaudio.transforms.TimeMasking(40)
# compound scaling - balcnes width depth resolution
# pretrained= model already knows tosee shapes n textures
# inchnas = accepts singlechannel instead of rgb
        self.backbone = timm.create_model(
            "tf_efficientnet_b0",
            pretrained=True,
            in_chans=1,
            num_classes=len(GENRES)
        )

    
    def forward(self,x):
        x = self.mel(x)
        x = torch.log(x+1e-6)

        x = (x - x.mean(dim=(2,3),keepdim=True)) / (x.std(dim=(2,3),keepdim=True)+1e-6)

        if self.training:
            x = self.freq_mask(x)
            x = self.time_mask(x)

        return self.backbone(x)

### EfficientNet Training

In [ ]:
run=wandb.init(
    project=WANDB_PROJECT,
    name="efficientnet_augmented",
    reinit=True,
    config={
        "model":"EfficientNetB0",
        "augmentation":"mixup + specaugment",
        "optimizer":"AdamW",
        "epochs":EPOCHS
    }
)

In [ ]:
effnet_model = EfficientNetAudio().to(device)
optimizer = torch.optim.AdamW(effnet_model.parameters(),lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=30)
criterion = nn.CrossEntropyLoss()

for epoch in range(30):
    effnet_model.train()
    total_loss = 0

    for x,y in tqdm(train_loader):
        x,y = x.to(device),y.to(device)
        optimizer.zero_grad()
        preds = effnet_model(x)
        
        loss = criterion(preds,y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    run.log({
        "epoch": epoch,
        "train_loss": avg_loss,
        "lr": optimizer.param_groups[0]["lr"]
    })
    scheduler.step()

    print("Epoch",epoch,"Loss",total_loss/len(train_loader))

### EfficientNet Evalution

In [ ]:
f1, acc = evaluate_model(effnet_model)

run.log({
    "F1_score": f1,
    "accuracy": acc
})
run.finish()

## Final EfficientNet Model
### Enhanced Audio Classifier

In [ ]:
class AudioClassifier(nn.Module):

    def __init__(self):
        super().__init__()
        self.mel = mel_transform
        self.freq_mask = torchaudio.transforms.FrequencyMasking(48)
        self.time_mask = torchaudio.transforms.TimeMasking(96)

        self.backbone = timm.create_model(
            "tf_efficientnet_b0",
            pretrained=True,
            in_chans=1,
            num_classes=len(GENRES)
        )

    def forward(self,x):
        x = self.mel(x)
        x = torch.log(x + 1e-6)

        x = (x - x.mean(dim=(2,3), keepdim=True)) / (x.std(dim=(2,3), keepdim=True) + 1e-6)

        if self.training:
            if random.random() < 0.5:
                x = self.freq_mask(x)
            if random.random() < 0.5:
                x = self.time_mask(x)

        return self.backbone(x)

### Final Model Architecture

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="efficientnet_final_augmented",
    reinit=True,
    config={
        "model": "EfficientNetB0",
        "augmentation": "Mixup + SpecAugment + T-Stretch",
        "epochs": EPOCHS,
        "lr": 3e-4
    }
)

In [ ]:
effnet_aug_model = AudioClassifier().to(device)
optimizer = torch.optim.AdamW(effnet_aug_model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)
# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
criterion = nn.CrossEntropyLoss()

### Final Model Training

In [ ]:
for epoch in range(EPOCHS):
    effnet_aug_model.train()
    total_loss = 0

    for x,y in tqdm(train_loader):

        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        lam = np.random.beta(0.4, 0.4)
        index = torch.randperm(x.size(0)).to(device)
        mixed_x = lam * x + (1 - lam) * x[index]
        
        y_a, y_b = y, y[index]
        preds = effnet_aug_model(mixed_x)
        
        loss = lam * criterion(preds, y_a) + (1 - lam) * criterion(preds, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(effnet_aug_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
    scheduler.step()
    
    run.log({"epoch": epoch, "loss": total_loss/len(train_loader)})

    print("Epoch",epoch,"Loss",total_loss/len(train_loader))

### Final Model Evaluation

In [ ]:
def evaluate():
    effnet_aug_model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x,y in val_loader:

            x = x.to(device)
            y = y.to(device)

            preds = effnet_aug_model(x)
            preds = preds.argmax(1)

            correct += (preds==y).sum().item()
            total += len(y)

    return correct/total

In [ ]:
val_acc = evaluate()
print("Validation Accuracy:",val_acc)
run.log({"final_val_accuracy": val_acc})
run.finish()

# Test Dataset Preparation

In [ ]:
class MashupDataset(Dataset):

    def __init__(self,test_df):
        self.df = test_df

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        path = os.path.join(DATA_DIR,row.filename)
        y,_ = librosa.load(path,sr=22050)
        y = torch.tensor(y).float().unsqueeze(0)

        return y,row.id

## Test Data Loader

In [ ]:
test_df = pd.read_csv(TEST_CSV)

test_ds = MashupDataset(test_df)

test_loader = DataLoader(
    test_ds,
    batch_size=16,
    # shuffle=False
    collate_fn=lambda b: collate_fn([(x,0) for x,_ in b])
)

# Ensemble Prediction
## Audio Prediction Function

In [ ]:
def predict_audio(audio):

    crop_size = 22050 * 6
    crops = []
    
    # for start in range(0, max(len(audio) - crop_size + 1, 1), crop_size // 6):
    #     crop = audio[start:start + crop_size]
    
    #     if len(crop) < crop_size:
    #         crop = np.pad(crop, (0, crop_size - len(crop)))
    #     crops.append(crop)

    for start in range(0, len(audio) - crop_size, crop_size // 4):
        crops.append(audio[start:start+crop_size])

    if len(crops) == 0:
        crops.append(random_crop(audio,crop_size))

    preds = []
    
    for crop in crops:
        x = torch.tensor(crop).float().unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            p1 = torch.softmax(cnn_model(x),1)
            p2 = torch.softmax(crnn_model(x),1)
            p3 = torch.softmax(effnet_model(x),1)
            p4 = torch.softmax(effnet_aug_model(x),1)
            p = (
                0.15*p1 +
                0.25*p2 +
                0.35*p3 +
                0.25*p4
            )

        preds.append(p.cpu().numpy())
    preds = np.mean(preds,axis=0)

    return preds.argmax()

## Ensemble Run Initialization

In [ ]:
ensemble_run = wandb.init(
    project=WANDB_PROJECT, 
    name="final_ensemble_submission",
    reinit=True,
)

## Generating Predictions

In [ ]:
cnn_model.eval()
crnn_model.eval()
effnet_model.eval()
effnet_aug_model.eval()

predictions = []

for i,row in tqdm(test_df.iterrows(), total=len(test_df)):

    path = os.path.join(DATA_DIR, row.filename)

    y,_ = librosa.load(path, sr=22050)

    pred = predict_audio(y)

    predictions.append(pred)
    
    if i % 100 == 0:
        ensemble_run.log({"progress": i / len(test_df)})

# Submission

In [ ]:
submission = pd.read_csv(SUBMISSION_CSV)

submission["genre"] = [IDX_TO_GENRE[p] for p in predictions]

submission.head()

In [ ]:
submission.to_csv("submission.csv",index=False)
ensemble_run.finish()